 **Flash Rank**: Ultra-lite & Super-fast Python library for search & retrieval re-ranking.

- **Ultra-lite**: No heavy dependencies. Runs on CPU with a tiny ~4MB reranking model.
- **Super-fast**: Speed depends on the number of tokens in passages and query, plus model depth.
- **Cost-efficient**: Ideal for serverless deployments with low memory and time requirements.
- **Based on State-of-the-Art Cross-encoders**: Includes models like ms-marco-TinyBERT-L-2-v2 (default), ms-marco-MiniLM-L-12-v2, rank-T5-flan, and ms-marco-MultiBERT-L-12.
- **Sleek Models for Efficiency**: Designed for minimal overhead in user-facing scenarios.

_Flash Rank is tailored for scenarios requiring efficient and effective reranking, balancing performance with resource usage._

**Model Options:**
- **Nano**: ~4MB, blazing fast model with competitive performance (ranking precision).
- **Small**: ~34MB, slightly slower with the best performance (ranking precision).
- **Medium**: ~110MB, slower model with the best zero-shot performance (ranking precision).
- **Large**: ~150MB, slower model with competitive performance (ranking precision) for 100+ languages.

In [19]:
import logging
logging.getLogger("httpx").setLevel(logging.WARNING)

from flashrank import Ranker
from langchain_groq import ChatGroq
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.document_loaders import PyPDFLoader
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_compressors import FlashrankRerank
from langchain_classic.retrievers import ContextualCompressionRetriever
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnableSequence, RunnableParallel, RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

In [2]:
loader = PyPDFLoader(r'papers/RAG-for-NLP_paper.pdf')
docs = loader.load()

text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
chunks = text_splitter.split_documents(docs)

In [3]:
vector_store = FAISS.from_documents(
    documents=chunks,
    embedding=HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
)

vector_store.save_local("faiss_db")

In [4]:
retriever = vector_store.as_retriever(search_type="mmr", search_kwargs={"k": 10, "lambda_mult": 0.5})
retriever

VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x00000242A7BF2870>, search_type='mmr', search_kwargs={'k': 10, 'lambda_mult': 0.5})

In [5]:
retrieved_docs = retriever.invoke("What are the two main components of the RAG architecture?")
retrieved_docs

[Document(id='c796d32b-8f7c-4170-8d54-7b68a8733184', metadata={'producer': 'pdfTeX-1.40.21', 'creator': 'LaTeX with hyperref', 'creationdate': '2021-04-13T00:48:38+00:00', 'author': '', 'keywords': '', 'moddate': '2021-04-13T00:48:38+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.14159265-2.6-1.40.21 (TeX Live 2020) kpathsea version 6.3.2', 'subject': '', 'title': '', 'trapped': '/False', 'source': 'papers/RAG-for-NLP_paper.pdf', 'total_pages': 19, 'page': 9, 'page_label': '10'}, page_content='paper, as well as HuggingFace for their help in open-sourcing code to run RAG models. The authors\nwould also like to thank Kyunghyun Cho and Sewon Min for productive discussions and advice. EP\nthanks supports from the NSF Graduate Research Fellowship. PL is supported by the FAIR PhD\nprogram.\nReferences\n[1] Payal Bajaj, Daniel Campos, Nick Craswell, Li Deng, Jianfeng Gao, Xiaodong Liu, Rangan\nMajumder, Andrew McNamara, Bhaskar Mitra, Tri Nguyen, Mir Rosenberg, Xia Song, Alina'),
 Docu

In [6]:
ranker = Ranker(
    model_name="ms-marco-MiniLM-L-12-v2",
    cache_dir="C:/Users/RadhaKrishna/Documents/Generative AI Tutorial/Advanced RAG"
    
)

compressor = FlashrankRerank(client=ranker, top_n=3)

compressed_docs = compressor.compress_documents(
    documents=retrieved_docs,
    query="What are the two main components of the RAG architecture?"
)

INFO:flashrank.Ranker:Downloading ms-marco-MiniLM-L-12-v2...
ms-marco-MiniLM-L-12-v2.zip: 100%|██████████| 21.6M/21.6M [00:16<00:00, 1.34MiB/s]


In [7]:
compressed_docs

[Document(metadata={'id': 8, 'relevance_score': np.float32(0.21965653), 'producer': 'pdfTeX-1.40.21', 'creator': 'LaTeX with hyperref', 'creationdate': '2021-04-13T00:48:38+00:00', 'author': '', 'keywords': '', 'moddate': '2021-04-13T00:48:38+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.14159265-2.6-1.40.21 (TeX Live 2020) kpathsea version 6.3.2', 'subject': '', 'title': '', 'trapped': '/False', 'source': 'papers/RAG-for-NLP_paper.pdf', 'total_pages': 19, 'page': 1, 'page_label': '2'}, page_content='present without additional training.\nOur results highlight the beneﬁts of combining parametric and non-parametric memory with genera-\ntion for knowledge-intensive tasks—tasks that humans could not reasonably be expected to perform\nwithout access to an external knowledge source. Our RAG models achieve state-of-the-art results\non open Natural Questions [29], WebQuestions [3] and CuratedTrec [2] and strongly outperform'),
 Document(metadata={'id': 5, 'relevance_score': np.float32(

In [8]:
compression_retriever = ContextualCompressionRetriever(base_compressor=compressor, base_retriever=retriever)
compression_retriever

ContextualCompressionRetriever(base_compressor=FlashrankRerank(client=<flashrank.Ranker.Ranker object at 0x00000242D7D8D520>, top_n=3, score_threshold=0.0, model=None, prefix_metadata=''), base_retriever=VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x00000242A7BF2870>, search_type='mmr', search_kwargs={'k': 10, 'lambda_mult': 0.5}))

In [9]:
def format_docs(retrieved_docs):
  context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
  return context_text

In [12]:
prompt = PromptTemplate.from_template("""
Answer ONLY from the provided context.

If the context does not contain the answer,
say exactly:
"I do not have relevant information in the context."

Do not use outside knowledge.
Do not speculate.

Context:
{context}

Question:
{question}

Answer:
"""
)

In [15]:
llm = ChatGroq(
    model='llama-3.3-70b-versatile',
    temperature=0.4
)

In [16]:
parallel_chain = RunnableParallel({
    'context': RunnableSequence(compression_retriever, RunnableLambda(format_docs)),
    'question': RunnablePassthrough()
})

main_chain = parallel_chain | prompt | llm | StrOutputParser()

In [20]:
print(main_chain.invoke("In the FEVER fact verification task, how does RAG retrieve evidence differently from most other approaches, and what accuracy does it achieve?"))

In the FEVER fact verification task, RAG retrieves evidence differently from most other approaches in that it does not use supervision on retrieved evidence. The accuracy of RAG in retrieving evidence is such that the top retrieved document is from a gold article in 71% of cases, and a gold article is present in the top 10 retrieved articles in 90% of cases.


In [21]:
print(main_chain.invoke("What is the weather in Volcano, CA according to the RAG paper?"))

I do not have relevant information in the context.
